# Combinatorial AAV payload library: design, screen, and visualise

Designing a gene therapy vector means choosing from a menu of interchangeable
regulatory parts — promoter, transgene, polyadenylation signal. Each combination
must be screened for problem sequences before synthesis. The difficulty is that
some sites are invisible until two specific parts are placed next to each other:
the recognition sequence is split across the junction and present in neither part
alone.

Gen's graph model stores all combinations as paths through a single graph, so one
`search()` call screens every path simultaneously — including every junction.

**Scenario:** An AAV9 vector for limb-girdle muscular dystrophy type 2D (LGMD2D)
caused by loss of α-sarcoglycan (SGCA). The manufacturing protocol linearises the
transfer plasmid with **SbfI** (`CCTGCAGG`) before packaging; any internal SbfI
site would shatter the payload at this step.

- 3 promoters: MCK, CK8e, CAG
- 2 SGCA coding sequences: human codon-optimised, CpG-deoptimised
- 2 polyA signals: bGH, SV40
- **12 combinations** — but some are blocked by a junction-emergent SbfI site

In [1]:
import itertools
import pathlib
import tempfile

import gen
import pandas as pd

WORK_DIR = pathlib.Path(tempfile.mkdtemp())
repo = gen.Repository(str(WORK_DIR))

## Define parts

Each part carries a `metadata` dict with its `role` in the vector. The role will
drive the colour palette when we call `plot(colors=...)` — all promoters share one
colour, all transgenes another — so the graph's fork-and-rejoin topology is
immediately legible.

The fixed ITR flanks are included as single-element columns so every path through
the graph is a complete, self-contained AAV payload.

For transgene parts, `annotation_start` pins the annotated region to the ATG start
codon, trimming the Kozak leader from the CDS annotation. The full sequence
(including Kozak) is still stored in the graph and participates in junction
screening; only the annotation boundary moves.

In [2]:
sp = gen.SequencePart  # shorthand

# ITR flanks (abbreviated; real AAV2 ITRs are ~145 nt).
itr_left = [
    sp("itr_left", "CACTCCCTCTCTGCGCGCTCGCTCGCTCACTGAGGCCGGGCGACCAAAGGTCGCCCGACGCCCGGGCTTTGCCCGGGCGGCCTCAGTGAGCGAGCGAGCGCGCAGAG",
       metadata={"role": "itr"}),
]
itr_right = [
    sp("itr_right", "CTCTGCGCGCTCGCTCGCTCACTGAGGCCACCCGACCAAAGGTCGCCCGACGCCCGGGCTTTGCCCGGGCGGCCTCAGTGAGCGAGCGAGCGCGCAGAGAGGGAGTG",
       metadata={"role": "itr"}),
]

# Three promoters.
# CK8e ends ...CCTGCA — the first six bases of the SbfI octamer.
promoters = [
    sp("MCK",  "AGGCGGGAAGATGGATCCCCTTGAGCAGCTCGAGAGCCTCGAGATGATCCCTTGATCC",
       metadata={"role": "promoter"}),
    sp("CK8e", "TGAAGTGATCCTTGAGCAGCTCGAGAGCCTCGAGATGATCCCTTGGCAGCTAGTCAGCCAGTGCCTGCA",
       metadata={"role": "promoter"}),
    sp("CAG",  "GACATTGATTATTGACTAGTTATTAATAGTAATCAATTACGGGGTCATTAGTTCATAGCAGATCT",
       metadata={"role": "promoter"}),
]

# Two SGCA coding sequences.
# sgca_human_coopt carries a 7-nt Kozak leader (GGCCACC) before the ATG.  The
# leading GG also pairs with the terminal CCTGCA of CK8e to complete the SbfI
# octamer CCTGCAGG.  annotation_start=7 trims the Kozak from the CDS annotation
# so the annotated region begins at the ATG start codon.
# sgca_deopt starts directly at ATG, so no offset is needed.
transgenes = [
    sp("sgca_human_coopt",
       "GGCCACCATGGCGCAGGTCCTGGAGCTGCTGGAGAAGCTGCAGAAGCAGAAGATCGTCATGGACGAGCTGGACGAACTTCAGCTGTGA",
       metadata={"role": "transgene"},
       annotation_start=7),
    sp("sgca_deopt",
       "ATGCAAGTGCTGGAGCTGCTGGAGAAGCTGCAGAAGCAGAAGATCGTCATGGACGAGCTGGACGAGCTTCAGTTGTGA",
       metadata={"role": "transgene"}),
]

# Two polyadenylation signals.
polya = [
    sp("bGH_polyA",
       "TGTATTTGTTTTTTTGTATAGCATAGATGATAATATTTCAGGGCCCAGACATGATAAGATACATTGATGAGTTTGG",
       metadata={"role": "polya"}),
    sp("SV40_polyA",
       "AGATCTGAATTTTTGTTTTTATTTGTTTTATTTTTTAATTTAAAATAAATATTATTTTTAAATATTATTTTATTTTTTAATTT",
       metadata={"role": "polya"}),
]

parts_list = [itr_left, promoters, transgenes, polya, itr_right]
n_combinations = 1
for col in parts_list:
    n_combinations *= len(col)
print(f"{' × '.join(str(len(c)) for c in parts_list)} = {n_combinations} combinations")

1 × 3 × 2 × 2 × 1 = 12 combinations


## Import the library

`import_library` accepts a list of columns; each column is a list of
`SequencePart` objects. Every combination of one part per column becomes a
distinct path through the resulting graph — 12 paths stored compactly in a
single graph structure.

In [3]:
repo.import_library("LGMD2D-AAV9", parts_list)

sgs = repo.get_sequence_graphs()
sg = next(s for s in sgs if s.name == "LGMD2D-AAV9")
print(sg)

SequenceGraph(c159cfbe11de9ec31f7182354c512ff819588a7ee2edc2c7e54005cd00183751, collection="default", sample="reference", name="LGMD2D-AAV9")


## Visualise the library graph

The `colors` parameter accepts a callable that receives each stored `Annotation`
and returns a CSS hex colour (or `None` to suppress it). Here we map the `role`
field from each part's metadata to a fixed palette — all promoters in blue, all
transgenes in green, polyA in amber, and ITRs in grey — so the graph's
fork-and-rejoin topology is immediately legible without reading part names.

In [4]:
ROLE_COLORS = {
    "itr":       "#aaaaaa",
    "promoter":  "#4393c3",
    "transgene": "#1a9850",
    "polya":     "#e08214",
}

fig = sg.plot(
    rows=18,
    colors=lambda ann: ROLE_COLORS.get((ann.metadata or {}).get("role")),
)


fig

## Build the design matrix

Enumerate all combinations explicitly so search results can be mapped back to
part identities. This is the factorial design before any screening filter.

In [5]:
design = pd.DataFrame(
    itertools.product(
        [p.name for p in promoters],
        [t.name for t in transgenes],
        [a.name for a in polya],
    ),
    columns=["promoter", "transgene", "polya"],
)
design["construct_id"] = range(1, len(design) + 1)
design

,promoter,transgene,polya,construct_id
0,MCK,sgca_human_coopt,bGH_polyA,1
1,MCK,sgca_human_coopt,SV40_polyA,2
2,MCK,sgca_deopt,bGH_polyA,3
3,MCK,sgca_deopt,SV40_polyA,4
4,CK8e,sgca_human_coopt,bGH_polyA,5
5,CK8e,sgca_human_coopt,SV40_polyA,6
6,CK8e,sgca_deopt,bGH_polyA,7
7,CK8e,sgca_deopt,SV40_polyA,8
8,CAG,sgca_human_coopt,bGH_polyA,9
9,CAG,sgca_human_coopt,SV40_polyA,10


## Screen the full library for SbfI

A single `search()` call traverses all 12 paths simultaneously, including
every inter-part junction. SbfI (`CCTGCAGG`) would linearise the transfer
plasmid and shatter the payload during packaging.

In [6]:
sbfi_matches = sg.search("CCTGCAGG", "dna")
print(f"SbfI sites found across the full library: {len(sbfi_matches)}")

SbfI sites found across the full library: 2


## Diagnose: the site is junction-emergent

The match count is surprising — neither CK8e nor sgca_human_coopt carries SbfI
on its own. The recognition sequence is split across their junction:

- CK8e ends with `...CCTGCA` — the first 6 bases of the SbfI octamer
- sgca_human_coopt begins with `GGCCACC...` — the Kozak consensus, supplying `GG`

Together they complete `CCTGCA·GG` = `CCTGCAGG`. This site is invisible to any
PCR screen of isolated parts; it only surfaces when the full junction sequence is
interrogated.

In [7]:
def count_sbfi(seq: str) -> int:
    site = "CCTGCAGG"
    return sum(1 for i in range(len(seq) - len(site) + 1) if seq[i:i+len(site)] == site)

print("SbfI in individual parts:")
for part in promoters + transgenes:
    n = count_sbfi(part.sequence)
    flag = " ← !" if n else ""
    print(f"  {part.name:<22} {n} site(s){flag}")

print()
junction = promoters[1].sequence + transgenes[0].sequence  # CK8e + sgca_human_coopt
print(f"CK8e ends:               ...{promoters[1].sequence[-12:]}")
print(f"sgca_human_coopt starts:    {transgenes[0].sequence[:12]}...")
print(f"SbfI at CK8e+sgca_human_coopt junction: {count_sbfi(junction)}")

SbfI in individual parts:
  MCK                    0 site(s)
  CK8e                   0 site(s)
  CAG                    0 site(s)
  sgca_human_coopt       0 site(s)
  sgca_deopt             0 site(s)

CK8e ends:               ...CCAGTGCCTGCA
sgca_human_coopt starts:    GGCCACCATGGC...
SbfI at CK8e+sgca_human_coopt junction: 1


## Highlight the SbfI site on the graph

Navigate to the site and mark it in red. The graph makes the junction context
visible: the site straddles the CK8e→transgene edge — present when those two
nodes are adjacent on a path, absent on every other path through the graph.

In [8]:
fig2 = sg.plot(
    rows=18,
    colors=lambda ann: ROLE_COLORS.get((ann.metadata or {}).get("role")),
)

sbfi_matches = sg.search("CCTGCAGG", "dna")
print(f"SbfI sites found across the full library: {len(sbfi_matches)}")

for match in sbfi_matches:
    fig2.show(match, color="#d62728")
fig2

SbfI sites found across the full library: 2


## Flag blocked constructs in the design matrix

The SbfI site occurs at every CK8e × sgca_human_coopt junction, regardless
of polyA choice — so two of the twelve constructs are blocked.

In [9]:
design["blocked"] = (
    (design["promoter"] == "CK8e") & (design["transgene"] == "sgca_human_coopt")
)

clean = design[~design["blocked"]].reset_index(drop=True)
blocked = design[design["blocked"]]

print(f"{len(blocked)} blocked construct(s):")
print(blocked[["construct_id", "promoter", "transgene", "polya"]].to_string(index=False))
print()
print(f"{len(clean)} clean construct(s) proceed to synthesis.")
design.style.apply(
    lambda row: ["background-color: #fdd" if row.blocked else "" for _ in row],
    axis=1,
)

2 blocked construct(s):
 construct_id promoter        transgene      polya
            5     CK8e sgca_human_coopt  bGH_polyA
            6     CK8e sgca_human_coopt SV40_polyA

10 clean construct(s) proceed to synthesis.


,promoter,transgene,polya,construct_id,blocked
0,MCK,sgca_human_coopt,bGH_polyA,1,False
1,MCK,sgca_human_coopt,SV40_polyA,2,False
2,MCK,sgca_deopt,bGH_polyA,3,False
3,MCK,sgca_deopt,SV40_polyA,4,False
4,CK8e,sgca_human_coopt,bGH_polyA,5,True
5,CK8e,sgca_human_coopt,SV40_polyA,6,True
6,CK8e,sgca_deopt,bGH_polyA,7,False
7,CK8e,sgca_deopt,SV40_polyA,8,False
8,CAG,sgca_human_coopt,bGH_polyA,9,False
9,CAG,sgca_human_coopt,SV40_polyA,10,False


## Experimental design for the in vivo study

With 10 clean constructs from a 3×2×2 factorial (minus the 2 blocked cells),
the first question before committing to animal work is whether all part effects
are still independently estimable from the remaining design.

In [10]:
import numpy as np

def rank_of(df, formula_cols):
    """One-hot encode formula_cols and return the rank of the resulting matrix."""
    dummies = pd.get_dummies(df[formula_cols], drop_first=True, dtype=float)
    X = np.column_stack([np.ones(len(dummies)), dummies.values])
    return np.linalg.matrix_rank(X), X.shape[1]

rank, n_params = rank_of(clean, ["promoter", "transgene", "polya"])
print(f"Full clean design: {len(clean)} constructs, {n_params} parameters, rank {rank}")
print("  →", "all main effects estimable" if rank == n_params else "rank-deficient — review design")

# Pilot: fix bGH polyA, test all 5 remaining promoter × transgene combinations.
pilot = clean[clean.polya == "bGH_polyA"].reset_index(drop=True)
rank_p, n_params_p = rank_of(pilot, ["promoter", "transgene"])
print(f"\nPilot (bGH only): {len(pilot)} constructs, {n_params_p} parameters, rank {rank_p}")
print("  →", "promoter + transgene effects estimable" if rank_p == n_params_p else "rank-deficient")
pilot[["construct_id", "promoter", "transgene", "polya"]]

Full clean design: 10 constructs, 5 parameters, rank 5
  → all main effects estimable

Pilot (bGH only): 5 constructs, 4 parameters, rank 4
  → promoter + transgene effects estimable


,construct_id,promoter,transgene,polya
0,1,MCK,sgca_human_coopt,bGH_polyA
1,3,MCK,sgca_deopt,bGH_polyA
2,7,CK8e,sgca_deopt,bGH_polyA
3,9,CAG,sgca_human_coopt,bGH_polyA
4,11,CAG,sgca_deopt,bGH_polyA


## Export clean constructs for synthesis

`export_fasta` writes all paths for the requested sample. After the SbfI filter,
only the 10 clean constructs are exported — the blocked CK8e×sgca_human_coopt
paths were never assigned to a synthesis sample.

In [11]:
out_fasta = str(WORK_DIR / "LGMD2D_AAV9_v1.fa")
repo.export_fasta(out_fasta)
print(f"Exported to {out_fasta}")
with open(out_fasta) as f:
    headers = [l.strip() for l in f if l.startswith(">")]
print(f"{len(headers)} sequence(s) exported:")
for h in headers:
    print(" ", h)

Exported to /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/tmp4qy_bwy9/LGMD2D_AAV9_v1.fa
Exported to file /var/folders/f8/8zf8xczs0pxf9_vfnlmqlx9h0000gn/T/tmp4qy_bwy9/LGMD2D_AAV9_v1.fa
1 sequence(s) exported:
  >LGMD2D-AAV9
